### Financial Inclusion — Feature Engineering & Encoding (Train)

---

Gleiche Encoding-Strategie wie in `Model_Encoding.ipynb`, angewendet auf `Train.csv`.

| Column | Action | Method | Reason |
|---|---|---|---|
| `year` | ❌ Drop | — | No predictive signal; all data is 2016–2018 |
| `uniqueid` | ❌ Drop | — | Identifier, not a feature |
| `household_size` | ❌ Drop | — | Weak proxy; correlated with other features |
| `relationship_with_head` | ❌ Drop | — | Too granular; correlates with gender/age |
| `marital_status` | ❌ Drop | — | Many 'Dont know' values; limited signal |
| `country` | ✅ Keep | One-Hot Encoding | Nominal — no natural order across 4 countries |
| `location_type` | ✅ Keep | Binary (0/1) | Only 2 values: Rural=0, Urban=1 |
| `cellphone_access` | ✅ Keep | Binary (0/1) | Only 2 values: No=0, Yes=1 |
| `gender_of_respondent` | ✅ Keep | Binary (0/1) | Only 2 values: Female=0, Male=1 |
| `age_of_respondent` | ✅ Keep | Keep as-is (numeric) | Already numeric and continuous |
| `education_level` | ✅ Keep | Ordinal Encoding | Clear progression from no education → tertiary |
| `job_type` | ✅ Keep | One-Hot Encoding | No meaningful order across 10 job categories |
| `bank_account` | ✅ Keep | Binary (0/1) | **Target variable** — No=0, Yes=1 |

---

### Step 0 — Imports

In [1]:
import pandas as pd
import numpy as np

---
### Step 1 — Load the Raw Data

In [2]:
df = pd.read_csv('data/Train.csv')

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

Loaded: 23,524 rows × 13 columns


,country,year,uniqueid,bank_account,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,Kenya,2018,uniqueid_1,Yes,Rural,Yes,3,24,Female,Spouse,Married/Living together,Secondary education,Self employed
1,Kenya,2018,uniqueid_2,No,Rural,No,5,70,Female,Head of Household,Widowed,No formal education,Government Dependent
2,Kenya,2018,uniqueid_3,Yes,Urban,Yes,5,26,Male,Other relative,Single/Never Married,Vocational/Specialised training,Self employed


---
### Step 2 — Drop Columns

**Dropped:**
- `year` — all surveys were 2016–2018; no meaningful signal
- `uniqueid` — row identifier, not a feature
- `household_size` — weak proxy; adds noise without clear signal
- `relationship_with_head` — overlaps with age/gender; too granular
- `marital_status` — many 'Dont know' values; limited ML signal

In [3]:
COLS_TO_DROP = [
    'year',
    'uniqueid',
    'household_size',
    'relationship_with_head',
    'marital_status',
]

df = df.drop(columns=COLS_TO_DROP)

print(f'Remaining columns ({len(df.columns)}):')
for col in df.columns:
    print(f'  • {col}')

Remaining columns (8):
  • country
  • bank_account
  • location_type
  • cellphone_access
  • age_of_respondent
  • gender_of_respondent
  • education_level
  • job_type


---
### Step 3 — Binary Encoding

Columns with exactly **2 categories** → converted to 0 or 1.

| Column | 0 | 1 |
|---|---|---|
| `location_type` | Rural | Urban |
| `cellphone_access` | No | Yes |
| `gender_of_respondent` | Female | Male |
| `bank_account` | No | Yes ← **target** |

In [4]:
# location_type: Rural=0, Urban=1
df['bin_location_type'] = df['location_type'].map({'Rural': 0, 'Urban': 1})

# cellphone_access: No=0, Yes=1
df['bin_cellphone_access'] = df['cellphone_access'].map({'No': 0, 'Yes': 1})

# gender_of_respondent: Female=0, Male=1
df['bin_gender'] = df['gender_of_respondent'].map({'Female': 0, 'Male': 1})

# bank_account: No=0, Yes=1  (target variable)
df['target_bank_account'] = df['bank_account'].map({'No': 0, 'Yes': 1})

# Verify — no NaN should appear if all values are mapped
checks = ['bin_location_type', 'bin_cellphone_access', 'bin_gender', 'target_bank_account']
for col in checks:
    nulls = df[col].isnull().sum()
    vals = df[col].value_counts().to_dict()
    print(f'{col}: {vals}  |  NaN: {nulls}')

bin_location_type: {0: 14343, 1: 9181}  |  NaN: 0
bin_cellphone_access: {1: 17454, 0: 6070}  |  NaN: 0
bin_gender: {0: 13877, 1: 9647}  |  NaN: 0
target_bank_account: {0: 20212, 1: 3312}  |  NaN: 0


---
### Step 4 — Ordinal Encoding: Education Level

Education has a **clear natural order** from lowest to highest.

| Level | Encoded value |
|---|---|
| No formal education | 0 |
| Primary education | 1 |
| Secondary education | 2 |
| Vocational/Specialised training | 3 |
| Tertiary education | 4 |
| Other/Dont know/RTA | 0 *(treated as unknown = lowest)* |

In [5]:
education_order = {
    'No formal education': 0,
    'Primary education': 1,
    'Secondary education': 2,
    'Vocational/Specialised training': 3,
    'Tertiary education': 4,
    'Other/Dont know/RTA': 0,
}

df['ord_education_level'] = df['education_level'].map(education_order)

print('Education encoding check:')
check = df.groupby('education_level')['ord_education_level'].first().reset_index()
check.columns = ['Original label', 'Encoded value']
check = check.sort_values('Encoded value')
print(check.to_string(index=False))
print(f'\nNaN count: {df["ord_education_level"].isnull().sum()}')

Education encoding check:
                 Original label  Encoded value
            No formal education              0
            Other/Dont know/RTA              0
              Primary education              1
            Secondary education              2
Vocational/Specialised training              3
             Tertiary education              4

NaN count: 0


---
### Step 5 — One-Hot Encoding: Country & Job Type

- **Country**: 4 Länder → 3 Dummy-Spalten (Kenya als Baseline)
- **Job Type**: 10 Kategorien → 9 Dummy-Spalten (`Dont Know/Refuse to answer` als Baseline)

In [6]:
# --- Country ---
country_dummies = pd.get_dummies(
    df['country'],
    prefix='ohe_country',
    drop_first=True   # Kenya is the baseline (dropped)
).astype(int)

print('Country dummy columns created:')
for col in country_dummies.columns:
    print(f'  • {col}  (1 = yes, 0 = no)')
print('  → Baseline (dropped): Kenya')

# --- Job Type ---
job_dummies = pd.get_dummies(
    df['job_type'],
    prefix='ohe_job',
    drop_first=True   # "Dont Know/Refuse to answer" is the baseline (dropped)
).astype(int)

print('\nJob type dummy columns created:')
for col in job_dummies.columns:
    print(f'  • {col}')

# Attach to main dataframe
df = pd.concat([df, country_dummies, job_dummies], axis=1)

print(f'\nDataFrame shape after adding dummies: {df.shape}')

Country dummy columns created:
  • ohe_country_Rwanda  (1 = yes, 0 = no)
  • ohe_country_Tanzania  (1 = yes, 0 = no)
  • ohe_country_Uganda  (1 = yes, 0 = no)
  → Baseline (dropped): Kenya

Job type dummy columns created:
  • ohe_job_Farming and Fishing
  • ohe_job_Formally employed Government
  • ohe_job_Formally employed Private
  • ohe_job_Government Dependent
  • ohe_job_Informally employed
  • ohe_job_No Income
  • ohe_job_Other Income
  • ohe_job_Remittance Dependent
  • ohe_job_Self employed

DataFrame shape after adding dummies: (23524, 25)


---
### Step 6 — Build the Final Clean Feature Table

Nur die encoded Spalten behalten. Die Zielspalte `target_bank_account` wird als letzte Spalte mitgeführt.

In [7]:
# Columns to keep in the final ML-ready table
numeric_cols = ['age_of_respondent']
binary_cols  = ['bin_location_type', 'bin_cellphone_access', 'bin_gender']
ordinal_cols = ['ord_education_level']
ohe_cols     = [c for c in df.columns if c.startswith('ohe_')]
target_col   = ['target_bank_account']

final_cols = numeric_cols + binary_cols + ordinal_cols + ohe_cols + target_col

df_model = df[final_cols].copy()

print(f'Final feature table: {df_model.shape[0]:,} rows × {df_model.shape[1]} columns')
print(f'\nColumn list ({len(df_model.columns)} total):')
for col in df_model.columns:
    dtype = str(df_model[col].dtype)
    print(f'  {col:<45} [{dtype}]')

Final feature table: 23,524 rows × 18 columns

Column list (18 total):
  age_of_respondent                             [int64]
  bin_location_type                             [int64]
  bin_cellphone_access                          [int64]
  bin_gender                                    [int64]
  ord_education_level                           [int64]
  ohe_country_Rwanda                            [int64]
  ohe_country_Tanzania                          [int64]
  ohe_country_Uganda                            [int64]
  ohe_job_Farming and Fishing                   [int64]
  ohe_job_Formally employed Government          [int64]
  ohe_job_Formally employed Private             [int64]
  ohe_job_Government Dependent                  [int64]
  ohe_job_Informally employed                   [int64]
  ohe_job_No Income                             [int64]
  ohe_job_Other Income                          [int64]
  ohe_job_Remittance Dependent                  [int64]
  ohe_job_Self employed          

---
### Step 7 — Final Checks

In [8]:
print('=== Final data quality check ===')
print(f'Shape         : {df_model.shape}')
print(f'Missing values: {df_model.isnull().sum().sum()}')
print(f'Data types    : {df_model.dtypes.value_counts().to_dict()}')
print()
print('Value ranges for numeric/ordinal columns:')
print(df_model[numeric_cols + ordinal_cols].describe().round(2))
print()
print('Target distribution (bank_account):')
vc = df_model['target_bank_account'].value_counts()
print(f'  No  (0): {vc.get(0, 0):,}  ({vc.get(0, 0)/len(df_model)*100:.1f}%)')
print(f'  Yes (1): {vc.get(1, 0):,}  ({vc.get(1, 0)/len(df_model)*100:.1f}%)')

=== Final data quality check ===
Shape         : (23524, 18)
Missing values: 0
Data types    : {dtype('int64'): 18}

Value ranges for numeric/ordinal columns:
       age_of_respondent  ord_education_level
count           23524.00             23524.00
mean               38.81                 1.20
std                16.52                 0.95
min                16.00                 0.00
25%                26.00                 1.00
50%                35.00                 1.00
75%                49.00                 2.00
max               100.00                 4.00

Target distribution (bank_account):
  No  (0): 20,212  (85.9%)
  Yes (1): 3,312  (14.1%)


In [9]:
print('First 5 rows of the model-ready feature table:')
df_model.head()

First 5 rows of the model-ready feature table:


,age_of_respondent,bin_location_type,bin_cellphone_access,bin_gender,ord_education_level,ohe_country_Rwanda,ohe_country_Tanzania,ohe_country_Uganda,ohe_job_Farming and Fishing,ohe_job_Formally employed Government,ohe_job_Formally employed Private,ohe_job_Government Dependent,ohe_job_Informally employed,ohe_job_No Income,ohe_job_Other Income,ohe_job_Remittance Dependent,ohe_job_Self employed,target_bank_account
0,24,0,1,0,2,0,0,0,0,0,0,0,0,0,0,0,1,1
1,70,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,26,1,1,1,3,0,0,0,0,0,0,0,0,0,0,0,1,1
3,34,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0
4,26,1,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0


---
### Step 8 — Save the Feature Table

In [10]:
output_path = 'features_encoded_train.csv'
df_model.to_csv(output_path, index=False)

print(f'Saved to: {output_path}')
print(f'   Rows   : {df_model.shape[0]:,}')
print(f'   Columns: {df_model.shape[1]} (davon 1 Zielspalte: target_bank_account)')

Saved to: features_encoded_train.csv
   Rows   : 23,524
   Columns: 18 (davon 1 Zielspalte: target_bank_account)
